# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Metadata object (not a dict or list)
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (tables), fields (columns), and their `@id`s.

In [ ]:
# Explore record sets and their fields using their `@id`s
from pprint import pprint

print("\nRecord Sets:")
for record_set in metadata.record_sets:
    print(f"- RecordSet `@id`: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields:`")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, data type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets (using their `@id`s)
record_sets = [rs.id for rs in metadata.record_sets]

dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set: {record_set_id}")

# As an example, use the first record set for further analysis
main_record_set_id = record_sets[0] if len(record_sets) > 0 else None
if main_record_set_id:
    print(f"\nColumns in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations such as removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

All analysis is referenced using field and record set `@id`s as per mlcroissant best practices.

In [ ]:
# Select a numeric field for analysis (replace with actual field `@id` from overview)
# For demonstration, let's automatically pick the first numeric field in the main record set
main_df = dataframes[main_record_set_id].copy() if main_record_set_id else pd.DataFrame()
numeric_field_id = None

if main_record_set_id:
    rs = next((rs for rs in metadata.record_sets if rs.id == main_record_set_id), None)
    numeric_fields = [f for f in rs.fields if f.data_type in ["Float", "Integer", "Number"]]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        # Use the field's label or raw id (column) for pandas operations
        field_col = numeric_field_id
        print(f"Selected numeric field '@id': {numeric_field_id}")
        if field_col in main_df.columns:
            main_df = main_df[pd.to_numeric(main_df[field_col], errors='coerce').notnull()]
            main_df[field_col] = main_df[field_col].astype(float)
            threshold = main_df[field_col].mean()  # as an example, use the mean as threshold
            filtered_df = main_df[main_df[field_col] > threshold]
            print(f"Filtered records with {field_col} > {threshold:.2f}:")
            display(filtered_df.head())

            # Normalize
            filtered_df[f"{field_col}_normalized"] = (filtered_df[field_col] - filtered_df[field_col].mean()) / filtered_df[field_col].std()
            print(f"Normalized {field_col} for filtered records:")
            display(filtered_df[[field_col, f"{field_col}_normalized"]].head())

            # Group by a groupable/categorical field, if found
            group_field = None
            str_fields = [f for f in rs.fields if f.data_type == 'Text' and f.id in main_df.columns]
            if str_fields:
                group_field = str_fields[0].id
                print(f"\nGrouped by field '@id': {group_field}")
                grouped_df = filtered_df.groupby(group_field)[field_col].mean().reset_index().rename(columns={field_col: 'mean_' + field_col})
                display(grouped_df.head())
            else:
                print("No suitable text (categorical) field found for grouping.")
        else:
            print(f"Field '@id': {field_col} not found in DataFrame columns: {main_df.columns.tolist()}")
    else:
        print("No numeric field found in the main record set.")
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If there is a group field, boxplot by group
    if 'group_field' in locals() and group_field and group_field in main_df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data to plot visualizations.")

## 6. Conclusion
Summarized key findings and observations from the dataset exploration.

- The dataset was loaded using the `mlcroissant` library from its Croissant schema.
- Main record sets, fields, and field `@id`s were explored programmatically.
- Simple statistical filtering, normalization, and grouping were conducted using field `@id`s, to ensure robust and reproducible analysis referencing the Croissant schema.
- Basic distributions and group differences were visualized.

You can adapt and extend this notebook by selecting additional record sets or fields and conducting more advanced analyses.

<br>For further information, consult the [FAIR2 dataset on SenScience](https://sen.science/doi/10.71728/senscience.vq4a-28xa).